In [1]:
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 1. 设定参数
SEQ_LENGTH = 50       # 假设每条RNA序列长度为50nt
MOTIF = "UAUAUA"      # 我们假设这是某种能增强翻译的调控Motif
NUM_SAMPLES = 1000    # 正负样本各生成1000条

def generate_rna(has_motif=False):
    """随机生成一条RNA序列"""
    bases = ['A', 'U', 'C', 'G']
    seq = ''.join(random.choices(bases, k=SEQ_LENGTH))

    if has_motif:
        # 如果是阳性样本，强行把 Motif 插入到序列的随机位置
        insert_pos = random.randint(0, SEQ_LENGTH - len(MOTIF))
        seq = seq[:insert_pos] + MOTIF + seq[insert_pos + len(MOTIF):]
    return seq

# 2. 生成数据集并存入 Pandas DataFrame (类似 R 语言的 data.frame)
data = []
for _ in range(NUM_SAMPLES):
    data.append({'sequence': generate_rna(has_motif=True), 'label': 1})  # 阳性样本，标签为1
for _ in range(NUM_SAMPLES):
    data.append({'sequence': generate_rna(has_motif=False), 'label': 0}) # 阴性样本，标签为0

df = pd.DataFrame(data)
# 打乱数据集顺序，防止模型“死记硬背”
df = df.sample(frac=1).reset_index(drop=True)

print("数据集前5行预览：")
print(df.head())

数据集前5行预览：
                                            sequence  label
0  CGCGGCUAUAUACUCGCUAUGCUAACUCGUCGCACGCCCGUCGUUC...      1
1  CCGGAAGUUGGUAUAUACGGACGGUUUAUGCACUUCAUUCGCGGAA...      1
2  GAAGAAUCGAACCGGUGUCGAAAAAAACGGACCUGUAUAUAGUGUU...      1
3  UCCUUGAUUGAUUAGCCUAUAUACUAUUGGGGAGAAGACCAUUGGC...      1
4  CGUGCUAACUCACUAGUCCCCCUCGUAAACGUUAAUCAUAAAGAGG...      0


In [2]:
def sequence_to_onehot(seq):
    """
    将 RNA 序列转化为数字矩阵。
    A -> [1,0,0,0], U -> [0,1,0,0], C -> [0,0,1,0], G -> [0,0,0,1]
    """
    mapping = {
        'A': [1, 0, 0, 0],
        'U': [0, 1, 0, 0],
        'C': [0, 0, 1, 0],
        'G': [0, 0, 0, 1]
    }
    # 把序列里的每个碱基变成4个数字，然后展平变成一个长一维数组 (50*4 = 200个数字)
    onehot = []
    for base in seq:
        onehot.extend(mapping[base])
    return np.array(onehot, dtype=np.float32)

# 把 Pandas 里的序列列全部转化为矩阵
X = np.array([sequence_to_onehot(seq) for seq in df['sequence']])
y = df['label'].values.astype(np.float32)

print(f"X (特征) 的形状: {X.shape}") # 预期: (2000, 200)，即2000条数据，每条200个特征
print(f"y (标签) 的形状: {y.shape}") # 预期: (2000,)

X (特征) 的形状: (2000, 200)
y (标签) 的形状: (2000,)


In [3]:
class RNADataset(Dataset):
    """定义一个标准的 PyTorch 数据集类"""
    def __init__(self, features, labels):
        # 把 Numpy 数组转化为 PyTorch 专属的 Tensor (张量)
        self.X = torch.tensor(features)
        self.y = torch.tensor(labels).unsqueeze(1) # 调整标签形状以匹配模型输出

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 划分训练集 (80%) 和测试集 (20%)
train_size = int(0.8 * len(df))
train_dataset = RNADataset(X[:train_size], y[:train_size])
test_dataset = RNADataset(X[train_size:], y[train_size:])

# DataLoader 负责按批次把数据喂给模型，batch_size=32 意味着每次学习32条序列
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [9]:
# ==========================================
# 修复版【模块四】：搭建 1D-CNN 卷积神经网络
# ==========================================
class MotifClassifierCNN(nn.Module):
    def __init__(self):
        super(MotifClassifierCNN, self).__init__()
        # 卷积层：扫描器，in_channels=4 (A,U,C,G), kernel_size=6 (滑动窗口大小)
        self.conv1 = nn.Conv1d(in_channels=4, out_channels=16, kernel_size=6)
        self.relu = nn.ReLU()
        # 全局最大池化：捕获最强信号，无视位置
        self.pool = nn.AdaptiveMaxPool1d(1)
        # 输出层：打分
        self.fc = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 【核心修复区】：正确的数据重塑与翻转
        # 1. 先恢复成 (Batch, 50个碱基, 每个碱基4个值) 的生物学结构
        x = x.view(-1, 50, 4)
        # 2. 将第1维(碱基长度)和第2维(4个值)翻转 -> (Batch, 4个值, 50个碱基)
        x = x.transpose(1, 2)

        # 接下来就是正常的滑动扫描了
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return self.sigmoid(x)

# 实例化修复后的模型
model = MotifClassifierCNN()
print("已成功切换为 1D-CNN 架构，并且修复了维度错位问题！")

已成功切换为 1D-CNN 架构，并且修复了维度错位问题！


In [10]:
# 1. 定义损失函数 (Loss) 和优化器 (Optimizer)
criterion = nn.BCELoss() # 二分类交叉熵损失函数，专门用来做 0/1 分类题
optimizer = optim.Adam(model.parameters(), lr=0.005) # Adam优化器，负责根据误差去调整模型的内部参数

EPOCHS = 15 # 让模型把数据集反反复复看15遍

print("🚀 开始训练模型...")
for epoch in range(EPOCHS):
    model.train() # 开启训练模式
    total_loss = 0

    # 每次从 DataLoader 里拿取 32 条数据 (batch_X) 和真实标签 (batch_y)
    for batch_X, batch_y in train_loader:
        # Step 1: 清空前一步的梯度（必须做）
        optimizer.zero_grad()

        # Step 2: 让模型去猜（前向传播）
        predictions = model(batch_X)

        # Step 3: 计算模型猜的与真实结果之间的差距（算误差）
        loss = criterion(predictions, batch_y)

        # Step 4: 把误差反向传回去（反向传播）
        loss.backward()

        # Step 5: 优化器根据误差调整内部参数
        optimizer.step()

        total_loss += loss.item()

    # --- 每训练完一轮，在测试集上看看效果 ---
    model.eval() # 开启评估模式
    correct = 0
    total = 0
    with torch.no_grad(): # 评估时不需要计算梯度，省内存
        for test_X, test_y in test_loader:
            outputs = model(test_X)
            # 如果输出概率 > 0.5，我们就认为它预测是阳性 (1)
            predicted = (outputs > 0.5).float()
            total += test_y.size(0)
            correct += (predicted == test_y).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}], 训练误差 Loss: {total_loss/len(train_loader):.4f}, 测试集准确率: {accuracy:.2f}%")

print("🎉 训练完成！如果准确率接近 100%，说明模型成功发现了 Motif 的规律。")

🚀 开始训练模型...
Epoch [1/15], 训练误差 Loss: 0.6460, 测试集准确率: 85.25%
Epoch [2/15], 训练误差 Loss: 0.4557, 测试集准确率: 97.00%
Epoch [3/15], 训练误差 Loss: 0.2295, 测试集准确率: 97.50%
Epoch [4/15], 训练误差 Loss: 0.1241, 测试集准确率: 97.75%
Epoch [5/15], 训练误差 Loss: 0.0846, 测试集准确率: 98.50%
Epoch [6/15], 训练误差 Loss: 0.0659, 测试集准确率: 98.00%
Epoch [7/15], 训练误差 Loss: 0.0555, 测试集准确率: 99.25%
Epoch [8/15], 训练误差 Loss: 0.0483, 测试集准确率: 99.25%
Epoch [9/15], 训练误差 Loss: 0.0433, 测试集准确率: 99.00%
Epoch [10/15], 训练误差 Loss: 0.0396, 测试集准确率: 99.00%
Epoch [11/15], 训练误差 Loss: 0.0369, 测试集准确率: 99.25%
Epoch [12/15], 训练误差 Loss: 0.0348, 测试集准确率: 99.25%
Epoch [13/15], 训练误差 Loss: 0.0329, 测试集准确率: 99.25%
Epoch [14/15], 训练误差 Loss: 0.0338, 测试集准确率: 99.25%
Epoch [15/15], 训练误差 Loss: 0.0318, 测试集准确率: 99.25%
🎉 训练完成！如果准确率接近 100%，说明模型成功发现了 Motif 的规律。
